# Neural Networks
NNs can be constructed using `torch.nn` package. `nn.Module` contains layers, and a method `forward(input)` that returns the output.

#### Training procdure for NNs
1. Define a NN with some learnable parameters (weights and biases)
2. Iterate over a dataset of inputs
3. Process input through the network
4. Compute the loss (how far is the output from being correct)
5. Propagate gradients back into the network's parameters
6. Update the weights of the Network, typically using a simple update rule: `weight = weight - learning_rate * gradient`.


#### Define the Network

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        self.conv1 = nn.Conv2d(1, 6, 5)
        # with kernel_size = 5: the dim of each of the channels is (32 - 5 + 1 = 28) 28 x 28
        # with 32x32 is the image dim, 5 is the kernel size 
        self.conv2 = nn.Conv2d(6, 16, 5)
        # an affine operation y = Wx + b
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 5*5 from image dimensions
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, input):
        # Convolution Layer C1: out_shape: (N, 6, 28, 28)
        # where N is the batch size
        c1 = F.relu(self.conv1(input))
        # Subsampling Layer S2: 2x2 grid, purely functional
        # meaning this layer does not have any parameters. and outputs a (N, 6, 14, 14)
        s2 = F.max_pool2d(c1, (2, 2))
        # Convolution Layer C3: out_shape -> (N, 16, 10, 10) Tensor
        c3 = F.relu(self.conv2(s2))
        # Subsampling Layer S4: 2x2 grid, purely functinal
        # out_shape -> (N, 16, 5, 5)
        s4 = F.max_pool2d(c3, 2)
        # Flatten operation, purely functional. out_shape -> (N, 400)
        s4 = torch.flatten(s4, 1)
        # Fully Connected Layer F5: out_shape -> (N, 120)
        f5 = F.relu(self.fc1(s4))
        # Fully Connected Layer F6: out_shape(N, 84)
        f6 = F.relu(self.fc2(f5))
        # Output Layer f6 output: out_shape -> (N, 10)
        output = self.fc3(f6)
        return output


net = Net()
print(net)







Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


Only `forward` function needs to be defined, the `backward` function is automatically defined using `autograd`.
The learnable parameters of the model are returned by `net.parameters()`

In [25]:
net_params = list(net.parameters())
print(len(net_params))
print(net_params[0].size()) # conv1's weight


10
torch.Size([6, 1, 5, 5])


Pass a random input of 32 x 32 to the network. The input of a NN is defined as (batch_size, num_channels, height, width).

In [26]:
in1 = torch.randn(1, 1, 32, 32)
out = net.forward(in1)
print(out)

tensor([[ 0.1348, -0.0118, -0.0090,  0.0098, -0.0602,  0.0898,  0.0731, -0.0038,
         -0.1160, -0.0170]], grad_fn=<AddmmBackward0>)


Zero the gradient buffers of the network parameters and perform backpropagation with a random initialization output


In [27]:
net.zero_grad()
out.backward(torch.randn(1, 10))

`torch.nn` only support mini-batches. For example `Conv2d` will take 4D tensor of size: `nSamples x nChannels x Height x Width`. For single sample, a fake batch dimension can be added with `input.unsqueeze(0)`

#### Recap of the important classes

* `torch.tensor` - A *multidimensional array* with support for `autograd` operations like `backward`.
* `nn.Module` - provides a *convenient way of encapsulating parameters*, with helpers for moving them to GPU, exporting and loading etc.
* `nn.Parameter` - A tensor which *automatically gets registered as an attribute to nn.Module*
* `autograd.Function` - Implements *forward and backward definitions of autograd function*. Each tensor operation creates atlest a single **Function** node that connects to operations that created a **Tensor** and *encodes its history*.


#### Loss Function
Loss function basically takes the output of the NN and the known target and evaluate how far is the actual target from NN output. 
There are several kinds of loss functions defined under the nn package. One of the simplest loss function is `nn.MSELoss`.

In [28]:
out1 = net(in1)
targ1 = torch.randn(1, 10)
criterion = nn.MSELoss()
loss = criterion(out1, targ1)
print(loss)

tensor(1.3800, grad_fn=<MseLossBackward0>)


When we follow `loss` using `grad_fn`, the computation graph looks like:

input -> Conv2d -> ReLU -> max_pool2d -> Conv2d -> ReLU -> max_pool2d -> flatten -> Linear -> ReLU 
->Linear -> ReLU -> Linear
-> MSELoss -> loss  

when `loss.backward()` is called on the loss??, the whole graph is differentiated wrt NN parameters (weights and biases). All tensors in the graph, which has `requires_grad = True` will have their `.grad` Tensor accumulated with the gradient. 

For illustrations, let's follow a few steps backward.

In [32]:
print(loss.grad_fn)
print(loss.grad_fn.next_functions)



((<AddmmBackward0 object at 0x000001BAE96FD030>, 0), (None, 0))


In [31]:
print(f'Target variable requires grad attribute: {targ1.requires_grad}')
print(f'NN output requires grad attribute: {out1.requires_grad}')

Target variable requires grad attribute: False
NN output requires grad attribute: True


Somehow, I am not able to see the grad_fn backward computation graph with `loss.grad_fn.next_functions`.

#### Backprop

All you have to do is to call `loss.backward()`. Remember to clear the existing gradients else gradients will be accumulated.

Let's look at conv1's bias gradient before and after backward. Since, we have not introducted an optimizer, we directly act on the model i.e. `net.zero_grad()`. With optimizer prefer `optim.zero_grad()`

In [34]:
net.zero_grad()

print('conv1.bias.grad berfore backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad berfore backward
None
conv1.bias.grad after backward
tensor([-0.0091, -0.0005,  0.0404,  0.0178, -0.0125,  0.0239])


#### Weight Update

The simplest rule used in practice is the Stochastic Gradient Descent (SGD).


weights = weights - learning_rate * gradient

Implementation

In [ ]:
learning_rate = 0.01
for f in net.parameters():
    with torch.no_grad():
        f -= f.grad * learning_rate

There are many different types of weight update rules such as SGD, Nesterov-SGD, Adam, RMSProp, etc. All of these and many more has been implemented in `torch.optim` package.

In [ ]:
import torch.optim as optim

# Create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop
optimizer.zero_grad() # zeros the gradient buffers
output = net(in1)
loss = criterion(out1, targ1)
loss.backward()
optimizer.step() # Does the update

### Activation Functions
I am trying to plot activation functions.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

